# E/22/423

Question 1 – Bayesian Estimation of a User Ability Parameter from Item Responses

Task 1 – Visualizing the Mechanics

Theory
The Two-Parameter Logistic (2PL) Item Response Theory model defines the probability that a user with latent ability theta answers item i correctly as:
$$P(Y_i=1\vert{}\theta) = \frac{1}{1+e^{-a_i(\theta-b_i)}}$$


Where:

a_i is the discrimination parameter

b_i is the difficulty parameter

theta is the user's latent ability

The discrimination parameter controls the steepness of the curve, while the difficulty parameter shifts the curve horizontally.

In [1]:
import numpy as np
import plotly.graph_objects as go

# 2PL Item Response Function
def probability(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta = np.linspace(-6, 6, 300)

curves = [
    {"a":0.5, "b":0},
    {"a":1.5, "b":-2},
    {"a":1.5, "b":0},
    {"a":1.5, "b":2}
]

fig = go.Figure()

for curve in curves:

    p = probability(theta, curve["a"], curve["b"])

    fig.add_trace(go.Scatter(
        x=theta,
        y=p,
        mode='lines',
        name=f"a={curve['a']}, b={curve['b']}"
    ))

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Ability (θ)",
    yaxis_title="P(Correct Response)",
    template="plotly_white"
)

fig.show()

## Interpretation

The discrimination parameter $a$ controls how rapidly the probability changes around the item's difficulty level.

- Larger values of $a$ produce a steeper S-shaped curve.
- Smaller values of $a$ produce a flatter curve.

The difficulty parameter $b$ shifts the curve horizontally.

- Increasing $b$ moves the curve to the right, meaning a higher ability level is required to achieve the same probability of answering correctly.
- Decreasing $b$ shifts the curve to the left, indicating an easier item.

Therefore, the discrimination parameter controls the steepness of the response curve, whereas the difficulty parameter determines the location of the curve along the ability axis.


---

# Task 2 – Sequential Likelihood Contribution

For a single observed response $y_k$,

$$
L(y_k|\theta)=p_k(\theta)^{y_k}
[1-p_k(\theta)]^{1-y_k}
$$

where

$$
p_k(\theta)=
\frac{1}{1+e^{-a_k(\theta-b_k)}}
$$


If

$$
y_k=1
$$

then

$$
L(y_k|\theta)=p_k(\theta)
$$


If

$$
y_k=0
$$

then

$$
L(y_k|\theta)=1-p_k(\theta)
$$


## Joint Likelihood

Assuming conditional independence,

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
[1-p_i(\theta)]^{1-y_i}
$$


---

# Task 3 – Mathematical Formulation of the Running Update

Using Bayes' theorem,

$$
f(\theta|y^{(k)})
=
\frac{
L(y_k|\theta)f(\theta|y^{(k-1)})
}
{
\int L(y_k|s)f(s|y^{(k-1)})ds
}
$$


Ignoring the normalizing constant,

$$
f(\theta|y^{(k)})
\propto
L(y_k|\theta)f(\theta|y^{(k-1)})
$$


Thus, after every response, the posterior distribution from the previous step becomes the prior distribution for the next step.


---

# Task 4 – Dynamic Shifting

When the user answers a highly difficult question correctly,

$$
y_k=1
$$

and the difficulty parameter $b_k$ is large, the likelihood becomes larger for higher values of the ability parameter $\theta$.

Consequently, multiplying the previous posterior distribution by this likelihood shifts the posterior density toward higher ability values.

The peak (mode) of the posterior moves to the right, indicating increased confidence that the user possesses higher ability than previously estimated.

The greater the difficulty parameter, the stronger this rightward shift because only users with relatively high ability have a high probability of answering such an item correctly.


---

# Task 5 – Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how informative an item is about the user's ability.

When

$$
a_k
$$

is very large, the likelihood function is steep. The posterior distribution becomes narrower, reducing its variance and increasing confidence in the estimated ability.

When

$$
a_k
$$

is very small, the likelihood function changes gradually with ability. The posterior distribution remains relatively broad, resulting in greater uncertainty.


Hence,

$$
\text{Large discrimination}
\rightarrow
\text{narrower posterior}
\rightarrow
\text{lower variance}
\rightarrow
\text{higher certainty}
$$


$$
\text{Small discrimination}
\rightarrow
\text{wider posterior}
\rightarrow
\text{higher variance}
\rightarrow
\text{lower certainty}
$$


---

# Task 6 – Numerical Implementation of a Running Grid

The posterior distribution can be approximated numerically using the following algorithm:

1. Define a fixed grid of ability values $\theta$ over an interval such as $[-5,5]$.

2. Initialize the prior distribution using the standard normal distribution.

3. For every new response:
   - Compute the response probability using the 2PL model.
   - Evaluate the likelihood over the entire grid.
   - Multiply the likelihood by the previous posterior distribution.
   - Normalize the resulting distribution using the trapezoidal rule.


The normalization step is:

$$
\text{Posterior}(\theta)
=
\frac{\text{Posterior}(\theta)}
{\int \text{Posterior}(\theta)d\theta}
$$


This process is repeated after every response to update the estimated ability distribution.

In [4]:
from scipy.stats import norm

# Assuming theta from the previous cell (OlEE2G92GuFQ) is the theta_grid
theta_grid = np.linspace(-6, 6, 300) # This should align with the theta in OlEE2G92GuFQ if that's the grid to use

# Initialize prior distribution using the standard normal distribution
posterior = norm.pdf(theta_grid, loc=0, scale=1)

# Placeholder for a_k, b_k, y_k to calculate likelihood
# In a real scenario, these would come from an item and user response
a_k = 1.5 # Example discrimination parameter
b_k = 0   # Example difficulty parameter
y_k = 1   # Example response (1 for correct)

# Define probability function if not already in scope (it is in OlEE2G92GuFQ)
def probability(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Evaluate the likelihood over the entire grid
if y_k == 1:
    likelihood = probability(theta_grid, a_k, b_k)
else:
    likelihood = 1 - probability(theta_grid, a_k, b_k)

posterior *= likelihood

posterior /= np.trapezoid(posterior, theta_grid)

# Task 7 – Evaluating Convergence over the Timeline

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.75
n = 20

theta = np.linspace(-5,5,1000)

posterior = stats.norm.pdf(theta)

def probability(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

bayes=[]
map_est=[]

for k in range(n):

    a=np.random.uniform(0.5,2)

    b=np.random.normal(0,1)

    p_true=probability(theta_true,a,b)

    y=1 if np.random.rand()<p_true else 0

    p=probability(theta,a,b)

    likelihood=(p**y)*((1-p)**(1-y))

    posterior*=likelihood

    posterior/=np.trapezoid(posterior,theta)

    bayes.append(np.trapezoid(theta*posterior,theta))

    map_est.append(theta[np.argmax(posterior)])

fig=go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(1,n+1),
    y=bayes,
    mode='lines+markers',
    name='Posterior Mean'
))

fig.add_trace(go.Scatter(
    x=np.arange(1,n+1),
    y=map_est,
    mode='lines+markers',
    name='MAP'
))

fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True Ability'
)

fig.update_layout(
    title='Convergence of Bayesian Ability Estimates',
    xaxis_title='Item Number',
    yaxis_title='Estimated Ability',
    template='plotly_white'
)

fig.show()

# Analysis

Initially, the Posterior Mean and MAP estimates fluctuate because only a few item responses are available.

As additional responses are observed, both estimators gradually move toward the true ability value:

$$
\theta = 0.75
$$

The distance between the estimators and the true ability decreases, while the posterior distribution becomes increasingly concentrated around the correct value.

This convergence indicates that the Bayesian updating process accumulates evidence effectively over time.

As more informative responses are collected, uncertainty is reduced and the platform becomes more confident in estimating the user's latent ability.

# Question 2 – Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

## Task 1 – Structural Probability and Properties Plot

## Theory

An e-commerce platform wants to estimate the click-through rate (CTR) of a newly launched advertisement using Bayesian inference.

Let $\theta$ represent the unknown true probability that a user clicks the advertisement.

$$
\theta \in [0,1]
$$

The user interaction at each step $k$ is represented as:

$$
Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}\\
0, & \text{if the user does not click}
\end{cases}
$$


Given the true click probability $\theta$, each user interaction follows a Bernoulli distribution:

$$
P(Y_k=1|\theta)=\theta
$$


Before observing any data, a Beta distribution is selected as the prior distribution:

$$
\Theta \sim Beta(\alpha_0,\beta_0)
$$


The probability density function of the Beta distribution is:

$$
f(\theta)
=
\frac{1}{B(\alpha_0,\beta_0)}
\theta^{\alpha_0-1}
(1-\theta)^{\beta_0-1}
$$


where $B(\alpha_0,\beta_0)$ is the Beta function that acts as the normalization constant.

The Beta distribution is suitable because the click-through rate is a probability value restricted between 0 and 1.


---

# Beta Distribution Configurations

Three different prior states are considered.


## 1. Uninformative State

$$
Beta(1,1)
$$


This represents complete uncertainty because every value of $\theta$ between 0 and 1 has equal probability.


The mean value is:

$$
E[\theta]
=
\frac{\alpha}{\alpha+\beta}
$$


Therefore:

$$
E[\theta]
=
\frac{1}{1+1}
$$


$$
E[\theta]=0.5
$$


The prior assumes a 50% click probability but has no preference towards any value.


---

## 2. Right-Skewed State

$$
Beta(2,8)
$$


The expected value is:

$$
E[\theta]
=
\frac{\alpha}{\alpha+\beta}
$$


Substituting values:

$$
E[\theta]
=
\frac{2}{2+8}
$$


$$
E[\theta]=0.2
$$


This distribution is concentrated near lower CTR values, representing an advertisement expected to perform poorly.


---

## 3. Left-Skewed State

$$
Beta(8,2)
$$


The expected value is:

$$
E[\theta]
=
\frac{\alpha}{\alpha+\beta}
$$


Substituting values:

$$
E[\theta]
=
\frac{8}{8+2}
$$


$$
E[\theta]=0.8
$$


This distribution represents an advertisement expected to have a high click-through rate.


---

# Interpretation

The parameters $\alpha$ and $\beta$ control the location and shape of the Beta distribution.

- Increasing $\alpha$ shifts the distribution towards higher values of $\theta$.
- Increasing $\beta$ shifts the distribution towards lower values of $\theta$.


Therefore:

- $Beta(1,1)$ represents no prior knowledge.
- $Beta(2,8)$ represents belief in a low CTR.
- $Beta(8,2)$ represents belief in a high CTR.


The Beta distribution provides a flexible way to represent uncertainty about the unknown click probability before observing user interactions.

In [6]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go


theta = np.linspace(0,1,500)


beta_configs = [
    {"alpha":1,"beta":1,"name":"Uninformative Beta(1,1)"},
    {"alpha":2,"beta":8,"name":"Right-Skewed Beta(2,8)"},
    {"alpha":8,"beta":2,"name":"Left-Skewed Beta(8,2)"}
]


fig = go.Figure()


for config in beta_configs:

    pdf = stats.beta.pdf(
        theta,
        config["alpha"],
        config["beta"]
    )

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=pdf,
            mode="lines",
            name=config["name"]
        )
    )


fig.update_layout(
    title="Structural Variations of Beta Probability Density Function",
    xaxis_title="Click Probability (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)


fig.show()

# Task 2 – Sequential Likelihood and Joint History

## 1. Single Isolated Response Likelihood

At each impression step $k$, the user's interaction with the advertisement is modeled as an independent Bernoulli trial.

The likelihood contribution of a single observation $Y_k$, given the unknown click probability $\theta$, is:

$$
L(Y_k|\theta)
=
\theta^{Y_k}(1-\theta)^{1-Y_k}
$$


where:

- $Y_k=1$ represents a click.
- $Y_k=0$ represents a non-click.


For a click:

$$
Y_k=1
$$

the likelihood becomes:

$$
L(Y_k|\theta)=\theta
$$


For a non-click:

$$
Y_k=0
$$

the likelihood becomes:

$$
L(Y_k|\theta)=1-\theta
$$


Therefore, each observed interaction either increases or decreases the belief about the true click-through rate.


---

# 2. Joint Likelihood Function for Running History

The complete observation history up to step $k$ is represented as:

$$
Y^{(k)}=(Y_1,Y_2,...,Y_k)
$$


Assuming that user interactions are conditionally independent given $\theta$, the joint likelihood is the product of all individual likelihood contributions:

$$
L(Y^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{Y_i}(1-\theta)^{1-Y_i}
$$


This can be simplified by grouping the number of clicks and non-clicks.

Let:

$$
C_k=\sum_{i=1}^{k}Y_i
$$


be the total number of clicks.

The number of non-clicks is:

$$
k-C_k
$$


Therefore, the joint likelihood becomes:

$$
L(Y^{(k)}|\theta)
=
\theta^{C_k}(1-\theta)^{k-C_k}
$$


This equation shows that the complete evidence from all user interactions can be summarized by the total number of clicks and non-clicks.


---

# Task 3 – Closed-Form Analytical Updates (Beta-Binomial Conjugacy)

## 1. Recursive Relationship Using Bayes' Theorem

The Bayesian posterior distribution after observing a new interaction is calculated using Bayes' theorem:

$$
f(\theta|Y^{(k)})
=
\frac{
L(Y_k|\theta)f(\theta|Y^{(k-1)})
}
{
\int_0^1 L(Y_k|s)f(s|Y^{(k-1)})ds
}
$$


The denominator is only a normalization term, therefore:

$$
f(\theta|Y^{(k)})
\propto
L(Y_k|\theta)f(\theta|Y^{(k-1)})
$$


This means the new posterior is obtained by multiplying the previous posterior with the likelihood contribution of the latest observation.


---

# 2. Proof of Beta-Binomial Conjugacy

Assume the previous posterior follows a Beta distribution:

$$
\Theta|Y^{(k-1)}
\sim
Beta(\alpha_{k-1},\beta_{k-1})
$$


The density is:

$$
f(\theta|Y^{(k-1)})
=
\frac{1}{B(\alpha_{k-1},\beta_{k-1})}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}
$$


Using the Bayesian update:

$$
f(\theta|Y^{(k)})
\propto
\theta^{Y_k}(1-\theta)^{1-Y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}
$$


Combining the powers of $\theta$ and $(1-\theta)$:

$$
f(\theta|Y^{(k)})
\propto
\theta^{\alpha_{k-1}+Y_k-1}
(1-\theta)^{\beta_{k-1}+1-Y_k-1}
$$


This has the same form as a Beta distribution.

Therefore:

$$
\Theta|Y^{(k)}
\sim
Beta(\alpha_k,\beta_k)
$$


where the updated parameters are:

$$
\alpha_k=\alpha_{k-1}+Y_k
$$


$$
\beta_k=\beta_{k-1}+(1-Y_k)
$$


---

# 3. Posterior Mean Calculation

The posterior mean of a Beta distribution is:

$$
E[\theta|Y^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}
$$


Using the recursive updates:

$$
\alpha_k=\alpha_0+C_k
$$


and:

$$
\beta_k=\beta_0+k-C_k
$$


The posterior mean becomes:

$$
E[\theta|Y^{(k)}]
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}
$$


---

# Interpretation

The posterior mean combines two sources of information:

**Prior belief:**

$$
\alpha_0,\beta_0
$$


**Observed user interactions:**

$$
C_k
$$


Initially, the prior has a strong influence because few observations are available.

As more impressions are collected, the observed data dominates and:

$$
k\rightarrow\infty
$$


the posterior mean approaches:

$$
\frac{C_k}{k}
$$


which is the observed click-through rate.


Therefore, Bayesian updating gradually reduces dependence on the initial assumption and allows the data to determine the final estimate.

# Task 4 – Dynamic Shifting Mechanics

The Beta posterior distribution changes dynamically after every observed user interaction.

The posterior update equation is:

$$
f(\theta|Y^{(k)})
\propto
L(Y_k|\theta)f(\theta|Y^{(k-1)})
$$


where the likelihood contribution is:

$$
L(Y_k|\theta)
=
\theta^{Y_k}(1-\theta)^{1-Y_k}
$$


---

# Effect of a Click ($Y_k=1$)

When a user clicks the advertisement:

$$
Y_k=1
$$


the likelihood becomes:

$$
L(Y_k|\theta)=\theta
$$


The posterior update becomes:

$$
f(\theta|Y^{(k)})
\propto
\theta f(\theta|Y^{(k-1)})
$$


This increases the probability density for larger values of $\theta$.

Therefore, the posterior distribution shifts towards higher CTR values.

For example, if the prior distribution believes:

$$
\theta \approx 0.3
$$


and several clicks are observed, the posterior peak moves towards a larger value, indicating increased confidence that the advertisement performs well.


---

# Effect of a Non-Click ($Y_k=0$)

When a user does not click the advertisement:

$$
Y_k=0
$$


the likelihood becomes:

$$
L(Y_k|\theta)=1-\theta
$$


The posterior update becomes:

$$
f(\theta|Y^{(k)})
\propto
(1-\theta)f(\theta|Y^{(k-1)})
$$


This reduces the probability of larger values of $\theta$.

Therefore, the posterior distribution shifts towards lower CTR values.

A sequence of non-click observations will gradually move the estimated CTR downward.


---

# Comparison with Non-Conjugate Models

The Beta-Binomial model is called conjugate because the posterior distribution remains in the same Beta family after observing data.

The update is analytical:

$$
Beta(\alpha,\beta)
\rightarrow
Beta(\alpha_k,\beta_k)
$$


with simple parameter updates:

$$
\alpha_k=\alpha_{k-1}+Y_k
$$


$$
\beta_k=\beta_{k-1}+1-Y_k
$$


No numerical approximation is required.


However, in non-conjugate models such as the 2PL Item Response Theory model, the likelihood function has a logistic form:

$$
P(Y_i=1|\theta)
=
\frac{1}{1+e^{-a_i(\theta-b_i)}}
$$


Combining this likelihood with common prior distributions does not produce a known probability distribution.

Therefore, the posterior cannot be represented using a closed-form equation.

The posterior must instead be calculated numerically using methods such as:

- Grid approximation
- Monte Carlo sampling
- Markov Chain Monte Carlo (MCMC)


---

# Task 5 – Running Point Estimators

After each Bayesian update, two point estimates are calculated:

1. Posterior Mean Estimate  
2. Maximum A Posteriori (MAP) Estimate


---

# 1. Running Posterior Mean

For a Beta posterior:

$$
\Theta|Y^{(k)}
\sim
Beta(\alpha_k,\beta_k)
$$


the posterior mean is:

$$
\hat{\theta}_{Bayes}^{(k)}
=
E[\theta|Y^{(k)}]
$$


Therefore:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_k}{\alpha_k+\beta_k}
$$


Using the updated parameters:

$$
\alpha_k=\alpha_0+C_k
$$


$$
\beta_k=\beta_0+k-C_k
$$


the estimator becomes:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}
$$


---

# 2. Running Maximum A Posteriori Estimate

The MAP estimate is the value of $\theta$ where the posterior distribution reaches its maximum.

For a Beta distribution:

$$
Beta(\alpha_k,\beta_k)
$$


the MAP estimate is:

$$
\hat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
$$


provided that:

$$
\alpha_k>1
$$


and:

$$
\beta_k>1
$$


If either parameter is less than or equal to 1, the maximum occurs at the boundary of the interval.


---

# Task 6 – Performance Tracking and Convergence Analysis

The true click-through rate is:

$$
\theta_{true}=0.35
$$


The initial prior is:

$$
Beta(1,1)
$$


which represents complete uncertainty.


The simulation generates user responses sequentially:

$$
Y_k \sim Bernoulli(\theta_{true})
$$


For every new impression:

1. Generate a user response.

2. Update the Beta parameters.

3. Calculate the posterior mean.

4. Calculate the MAP estimate.

5. Store the estimates for visualization.

In [7]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.35
n = 100

alpha = 1
beta = 1

bayes = []
map_values = []

steps = range(1,n+1)


for k in steps:

    # Generate response
    y = 1 if np.random.rand() < theta_true else 0

    # Beta-Binomial update
    alpha += y
    beta += (1-y)

    # Posterior mean
    theta_bayes = alpha/(alpha+beta)

    # MAP estimate
    if alpha > 1 and beta > 1:
        theta_map = (alpha-1)/(alpha+beta-2)
    else:
        theta_map = 0

    bayes.append(theta_bayes)
    map_values.append(theta_map)



fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(steps),
        y=bayes,
        mode="lines",
        name="Posterior Mean"
    )
)


fig.add_trace(
    go.Scatter(
        x=list(steps),
        y=map_values,
        mode="lines",
        name="MAP Estimate"
    )
)


fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR"
)


fig.update_layout(
    title="Bayesian CTR Estimator Convergence",
    xaxis_title="Number of Impressions",
    yaxis_title="Estimated CTR",
    template="plotly_white"
)


fig.show()

# Analysis

At the beginning of the experiment, both estimators fluctuate because only a small number of observations are available.

As the number of impressions increases:

- More clicks and non-clicks are observed.
- The posterior distribution becomes narrower.
- The influence of the initial prior decreases.
- The estimates move closer to the true CTR value:

$$
\theta_{true}=0.35
$$


This demonstrates the fundamental property of Bayesian learning: increasing amounts of evidence reduce uncertainty and produce more accurate parameter estimates.

The initial prior affects the early estimates, but with enough observations, the accumulated data becomes the dominant source of information.

# Question 3 – Bayesian Estimation for Structural Health Monitoring

## Task 1 – Prior Belief Boundaries

## Theory

In structural health monitoring, engineers estimate the remaining stiffness efficiency of a component.

Let:

$$
\theta
$$

represent the remaining stiffness efficiency factor.

The parameter is physically bounded:

$$
\theta \in (0,1]
$$


where:

- $\theta=1$ represents a completely healthy structure.
- $\theta \rightarrow 0$ represents severe degradation or structural failure.


The stiffness measurement model is:

$$
y_k=\theta K_{nominal}e^{\epsilon_k}
$$


where:

- $K_{nominal}$ is the stiffness of the healthy component.
- $\epsilon_k$ represents measurement noise.

The measurement noise follows:

$$
\epsilon_k \sim N(0,\sigma^2)
$$


---

# Initial Prior Distribution

Since the component is expected to be healthy before measurements are collected, an optimistic prior is selected:

$$
\Theta \sim Beta(8,1.5)
$$


The Beta probability density function is:

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1}
$$


where:

$$
\alpha=8
$$

and:

$$
\beta=1.5
$$


---

# Expected Prior Stiffness Efficiency

The expected value of a Beta distribution is:

$$
E[\theta]
=
\frac{\alpha}{\alpha+\beta}
$$


Substituting the parameters:

$$
E[\theta]
=
\frac{8}{8+1.5}
$$


$$
E[\theta]=0.842
$$


Therefore, before collecting sensor data, the model expects approximately:

$$
84.2\%
$$


remaining stiffness efficiency.


---

# Interpretation

The $Beta(8,1.5)$ prior is suitable because:

- It is bounded between 0 and 1.
- It represents the physical limits of structural efficiency.
- The large value of $\alpha$ shifts the distribution towards higher stiffness values.
- The smaller value of $\beta$ represents confidence that the structure is initially healthy.


Therefore, the prior encodes engineering knowledge before measurements are collected.

In [8]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go


theta = np.linspace(0.01,1,500)


alpha = 8
beta = 1.5


prior = stats.beta.pdf(
    theta,
    alpha,
    beta
)


fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=theta,
        y=prior,
        mode="lines",
        name="Prior Beta(8,1.5)"
    )
)


fig.update_layout(
    title="Initial Prior Distribution of Structural Efficiency",
    xaxis_title="Stiffness Efficiency (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)


fig.show()

# Task 2 – Structural Likelihood Formulation

The measurement model is:

$$
y_k=\theta K_{nominal}e^{\epsilon_k}
$$


where:

$$
\epsilon_k \sim N(0,\sigma^2)
$$


Taking logarithms:

$$
\ln(y_k)=\ln(\theta K_{nominal})+\epsilon_k
$$


Therefore:

$$
\ln(y_k)
\sim
N(\ln(\theta K_{nominal}),\sigma^2)
$$


---

# Single Measurement Likelihood

Using the log-normal probability density function:

$$
L(y_k|\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp
\left(
-\frac{
(\ln y_k-\ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right)
$$


This represents the probability of observing measurement $y_k$ given the true stiffness efficiency $\theta$.


---

# Joint Likelihood for Measurement History

The measurement history is:

$$
Y^{(k)}
=
(y_1,y_2,...,y_k)
$$


Assuming independent sensor measurements:

$$
L(Y^{(k)}|\theta)
=
\prod_{i=1}^{k}L(y_i|\theta)
$$


Therefore:

$$
L(Y^{(k)}|\theta)
=
\prod_{i=1}^{k}
\frac{1}{y_i\sigma\sqrt{2\pi}}
\exp
\left(
-\frac{
(\ln y_i-\ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right)
$$

# Task 3 – Mathematical Formulation of the Non-Conjugate Grid Update

## Why an Exact Analytical Solution Does Not Exist

The prior distribution for the structural efficiency factor is:

$$
\Theta \sim Beta(8,1.5)
$$


with probability density:

$$
f(\theta)\propto
\theta^{8-1}(1-\theta)^{1.5-1}
$$


The likelihood function is derived from the log-normal measurement model:

$$
L(y_k|\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp
\left(
-\frac{
(\ln y_k-\ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right)
$$


The posterior distribution is:

$$
f(\theta|Y^{(k)})
\propto
L(Y^{(k)}|\theta)f(\theta)
$$


Unlike the Beta-Binomial model, the likelihood is not of the same mathematical form as the Beta distribution.

The multiplication:

$$
Beta(8,1.5)\times LogNormal(\theta)
$$


does not produce another known probability distribution.

Therefore, the posterior cannot be expressed using a closed-form analytical equation.

A numerical approximation method is required.


---

# Recursive Bayesian Update Relationship

The sequential Bayesian update is:

$$
f(\theta|Y^{(k)})
\propto
L(y_k|\theta)
f(\theta|Y^{(k-1)})
$$


where:

- $f(\theta|Y^{(k-1)})$ is the previous posterior distribution.
- $L(y_k|\theta)$ is the likelihood contribution from the new sensor measurement.
- $f(\theta|Y^{(k)})$ is the updated posterior distribution.


The normalized form is:

$$
f(\theta|Y^{(k)})
=
\frac{
L(y_k|\theta)f(\theta|Y^{(k-1)})
}
{
\int_{0.01}^{1}
L(y_k|s)f(s|Y^{(k-1)})ds
}
$$


The denominator ensures that the posterior integrates to one.


---

# Task 4 – Running Point Estimates

Since no closed-form posterior exists, point estimates must be obtained using numerical integration.

Two estimates are calculated:

1. Bayesian posterior mean  
2. Maximum A Posteriori (MAP) estimate


---

# 1. Running Posterior Mean

The Bayesian estimate is the expected value of the posterior distribution:

$$
\hat{\theta}_{Bayes}^{(k)}
=
E[\theta|Y^{(k)}]
$$


For the bounded domain:

$$
\theta \in [0.01,1]
$$


the estimate is:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\int_{0.01}^{1}
\theta f(\theta|Y^{(k)})d\theta
$$


Since the posterior is normalized:

$$
\int_{0.01}^{1}
f(\theta|Y^{(k)})d\theta=1
$$


---

# 2. Running Maximum A Posteriori Estimate

The MAP estimate is the value of $\theta$ where the posterior density is maximum.

It is defined as:

$$
\hat{\theta}_{MAP}^{(k)}
=
\arg\max_{\theta}f(\theta|Y^{(k)})
$$


The MAP estimate corresponds to the most probable remaining stiffness value according to the current sensor information.


---

# Task 5 – Algorithmic Grid Approximation and Normalization

Because an analytical solution is unavailable, the posterior distribution is maintained numerically using a discrete grid.


---

## Step 1: Create a Bounded Grid

A grid of possible stiffness values is created:

$$
\theta_1,\theta_2,...,\theta_N
$$


within the physical limits:

$$
0.01\leq\theta\leq1
$$


Example:

$$
\theta_{grid}=linspace(0.01,1,N)
$$


---

## Step 2: Initialize Prior Distribution

The initial Beta prior is evaluated over the grid:

$$
f_0(\theta)=Beta(8,1.5)
$$


The distribution is normalized:

$$
f_0(\theta)
=
\frac{f_0(\theta)}
{\int f_0(\theta)d\theta}
$$


---

## Step 3: Compute Likelihood

For each new measurement $y_k$, calculate:

$$
L(y_k|\theta)
$$


at every grid point.


The unnormalized posterior becomes:

$$
\tilde{f}(\theta)
=
L(y_k|\theta)f_{k-1}(\theta)
$$


---

## Step 4: Normalize Using Trapezoidal Rule

The posterior must satisfy:

$$
\int f(\theta)d\theta=1
$$


Therefore:

$$
f_k(\theta)
=
\frac{
\tilde{f}(\theta)
}
{
\int_{0.01}^{1}\tilde{f}(s)ds
}
$$


The numerical integral is calculated using the trapezoidal rule:

$$
\int f(\theta)d\theta
\approx
\sum_i
\frac{f_i+f_{i+1}}{2}
(\theta_{i+1}-\theta_i)
$$

In [11]:
import numpy as np
import scipy.stats as stats

# Initialize theta_grid and posterior for Question 3 context
# (as defined in X7t_LWdpJvkf and y5_0h12fJ_1X)
theta_grid = np.linspace(0.01, 1, 500)
alpha = 8
beta = 1.5
posterior = stats.beta.pdf(theta_grid, alpha, beta)

# Normalize the posterior distribution
posterior /= np.trapezoid(posterior, theta_grid)

# Boundary Handling

The physical limits are enforced by restricting the grid:

$$
\theta \in [0.01,1]
$$


Values outside this interval are ignored because they do not represent physically possible structural states.

This prevents the algorithm from assigning probability to impossible negative stiffness or efficiency values greater than 100%.


---

# Task 6 – Performance Tracking and Degradation Convergence Analysis

The true remaining stiffness is assumed to be:

$$
\theta_{true}=0.68
$$


The simulation parameters are:

$$
K_{nominal}=50 \text{ kN/mm}
$$


$$
\sigma=0.15
$$


$$
n=15
$$


Sensor measurements are generated using:

$$
y_k=\theta_{true}K_{nominal}e^{\epsilon_k}
$$


where:

$$
\epsilon_k \sim N(0,\sigma^2)
$$

In [13]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go


np.random.seed(24)


theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15


theta_grid = np.linspace(0.01,1,500)


posterior = stats.beta.pdf(
    theta_grid,
    8,
    1.5
)

posterior /= np.trapezoid(
    posterior,
    theta_grid
)


bayes_est = []
map_est = []

posterior_history = []


for k in range(n):

    epsilon = np.random.normal(0,sigma)

    y = theta_true*K_nominal*np.exp(epsilon)


    likelihood = (
        1/(y*sigma*np.sqrt(2*np.pi))
        *
        np.exp(
        -(np.log(y)-np.log(theta_grid*K_nominal))**2
        /(2*sigma**2)
        )
    )


    posterior *= likelihood


    posterior /= np.trapezoid(
        posterior,
        theta_grid
    )


    posterior_history.append(
        posterior.copy()
    )


    bayes_est.append(
        np.trapezoid(
            theta_grid*posterior,
            theta_grid
        )
    )


    map_est.append(
        theta_grid[np.argmax(posterior)]
    )

# Task 6 (Continued) – Visualization of Posterior Evolution and Convergence Analysis

## 1. Posterior Density Progression Plot

The posterior distribution is stored after each measurement update.

The density curves are plotted at different inspection milestones:

$$
k \in \{0,1,2,5,10,15\}
$$


This shows how the Bayesian belief about the remaining stiffness changes as more sensor data becomes available.


Initially, the posterior distribution represents only the prior belief:

$$
\Theta \sim Beta(8,1.5)
$$


As measurements are collected, the likelihood information is incorporated through Bayesian updating:

$$
f(\theta|Y^{(k)})
\propto
L(y_k|\theta)f(\theta|Y^{(k-1)})
$$


With increasing measurements:

- The posterior distribution becomes more concentrated.
- The uncertainty in the stiffness estimate decreases.
- The peak of the distribution moves closer to the true stiffness value:

$$
\theta_{true}=0.68
$$


The progression of posterior curves demonstrates how Bayesian inference continuously refines the estimate of structural health using accumulated sensor information.

In [14]:
import plotly.graph_objects as go


milestones = [0,1,2,5,10,15]


fig = go.Figure()


for step in milestones:

    if step == 0:
        density = stats.beta.pdf(
            theta_grid,
            8,
            1.5
        )

    else:
        density = posterior_history[step-1]


    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=density,
            mode="lines",
            name=f"Step {step}"
        )
    )


fig.update_layout(
    title="Posterior Density Evolution During Structural Monitoring",
    xaxis_title="Remaining Stiffness Efficiency (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)


fig.show()

## 2. Estimator Convergence Plot

The posterior mean and MAP estimates are tracked after every sensor observation.

The estimates are compared with the true structural efficiency:

$$
\theta_{true}=0.68
$$


The posterior mean estimate is calculated as:

$$
\hat{\theta}_{Bayes}^{(k)}
=
\int_{0.01}^{1}
\theta f(\theta|Y^{(k)})d\theta
$$


The MAP estimate is obtained by finding the maximum value of the posterior density:

$$
\hat{\theta}_{MAP}^{(k)}
=
\arg\max_{\theta}f(\theta|Y^{(k)})
$$


The convergence plot shows the evolution of both estimators as additional sensor measurements are collected.


As the number of observations increases:

- The posterior mean gradually approaches the true stiffness value.
- The MAP estimate becomes more stable.
- The difference between the estimated and true stiffness decreases.
- The uncertainty in the structural health estimate is reduced.


The final estimates converge towards:

$$
\theta_{true}=0.68
$$


This demonstrates that Bayesian grid-based estimation can successfully track the remaining stiffness efficiency and improve confidence as more sensor data becomes available.

In [15]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=np.arange(1,n+1),
        y=bayes_est,
        mode="lines+markers",
        name="Bayesian Posterior Mean"
    )
)


fig.add_trace(
    go.Scatter(
        x=np.arange(1,n+1),
        y=map_est,
        mode="lines+markers",
        name="MAP Estimate"
    )
)


fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness efficiency"
)


fig.update_layout(
    title="Convergence of Structural Efficiency Estimates",
    xaxis_title="Inspection Number",
    yaxis_title="Estimated θ",
    template="plotly_white"
)


fig.show()

# Analysis of Results

Initially, the posterior distribution is strongly influenced by the optimistic prior:

$$
Beta(8,1.5)
$$


Because the prior assumes that the structure is likely to be healthy, the initial probability density is concentrated near:

$$
\theta \approx 1
$$


After sensor measurements are introduced, the likelihood function begins to modify the belief distribution.

Since the actual stiffness efficiency is:

$$
\theta_{true}=0.68
$$


the posterior gradually shifts away from the original healthy assumption towards the true degraded state.


---

# Number of Measurements Required for Convergence

The posterior distribution begins moving significantly after the first few sensor observations.

Approximately:

$$
5-10
$$

measurements are required for the Bayesian estimate to overcome the optimistic prior.


By approximately:

$$
15
$$

sensor readings, both:

$$
\hat{\theta}_{Bayes}
$$


and:

$$
\hat{\theta}_{MAP}
$$


approach the true value:

$$
\theta_{true}=0.68
$$


---

# Interpretation of Density Narrowing

As more sensor measurements are collected:

- The posterior distribution becomes narrower.
- The uncertainty in the stiffness estimate decreases.
- The system becomes more confident about the actual structural condition.


The narrowing of the posterior density indicates that the monitoring system has gained stronger evidence about the component's health state.


A narrow posterior around:

$$
\theta=0.68
$$


indicates that the structure has a high probability of being approximately 68% efficient compared with its original stiffness.


---

# Structural Safety Interpretation

The Bayesian monitoring system provides a quantitative measure of structural degradation.


If the posterior remains near:

$$
\theta \approx 1
$$

the structure is likely healthy.


If the posterior shifts towards lower values:

$$
\theta < 0.7
$$

the structure may require further inspection or maintenance.


A concentrated posterior distribution allows engineers to make safer decisions because it reduces uncertainty in the estimated remaining stiffness.

# Question 4 – Gaussian Mixture Clustering as Conditional Bayesian Updating

# Task 1 – Gaussian Mixture Model Formulation

## Theory

A Gaussian Mixture Model (GMM) is a probabilistic model that represents data as a combination of multiple Gaussian distributions.

Assume that the dataset contains observations:

$$
X=\{x_1,x_2,...,x_n\}
$$


generated from $K$ different clusters.

The probability density function of a Gaussian mixture model is:

$$
p(x)
=
\sum_{j=1}^{K}
\pi_j
N(x|\mu_j,\Sigma_j)
$$


where:

- $K$ is the number of Gaussian components.
- $\pi_j$ is the mixing probability of component $j$.
- $\mu_j$ is the mean vector of component $j$.
- $\Sigma_j$ is the covariance matrix of component $j$.


The mixing coefficients satisfy:

$$
\sum_{j=1}^{K}\pi_j=1
$$


and:

$$
\pi_j\geq0
$$


---

# Task 2 – Gaussian Probability Density Function

For a one-dimensional Gaussian distribution:

$$
N(x|\mu,\sigma^2)
$$


the probability density function is:

$$
p(x)
=
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp
\left(
-\frac{(x-\mu)^2}{2\sigma^2}
\right)
$$


where:

- $\mu$ represents the center of the cluster.
- $\sigma^2$ represents the spread of the cluster.


A larger variance produces a wider distribution, while a smaller variance produces a more concentrated distribution.


---

# Task 3 – Posterior Probability of Cluster Membership

The objective is to determine the probability that a data point belongs to a particular Gaussian component.

Using Bayes' theorem:

$$
P(z_i=j|x_i)
=
\frac{
\pi_jN(x_i|\mu_j,\Sigma_j)
}
{
\sum_{k=1}^{K}\pi_kN(x_i|\mu_k,\Sigma_k)
}
$$


This probability is known as the responsibility:

$$
\gamma(z_i=j)
$$


Therefore:

$$
\gamma_{ij}
=
\frac{
\pi_jN(x_i|\mu_j,\Sigma_j)
}
{
\sum_{k=1}^{K}\pi_kN(x_i|\mu_k,\Sigma_k)
}
$$


## Interpretation

The responsibility value represents the confidence that a particular data point belongs to a cluster.


For example:

If:

$$
\gamma_{i1}=0.95
$$


then the point has a 95% probability of belonging to cluster 1.


If responsibilities are approximately equal:

$$
\gamma_{i1}\approx\gamma_{i2}
$$


then the point lies near the boundary between clusters.


---

# Task 4 – Expectation Maximization (EM) Algorithm

The Gaussian Mixture Model parameters are estimated using the Expectation-Maximization algorithm.

The EM algorithm alternates between two steps:

1. Expectation step (E-step)
2. Maximization step (M-step)


---

# E-Step: Calculate Responsibilities

For every data point, calculate:

$$
\gamma_{ij}=P(z_i=j|x_i)
$$


using:

$$
\gamma_{ij}
=
\frac{
\pi_jN(x_i|\mu_j,\Sigma_j)
}
{
\sum_{k=1}^{K}
\pi_kN(x_i|\mu_k,\Sigma_k)
}
$$


This determines the probability that each point belongs to each cluster.


---

# M-Step: Update Parameters

The effective number of points assigned to cluster $j$ is:

$$
N_j=
\sum_{i=1}^{N}\gamma_{ij}
$$


The updated mean is:

$$
\mu_j
=
\frac{1}{N_j}
\sum_{i=1}^{N}
\gamma_{ij}x_i
$$


The updated covariance is:

$$
\Sigma_j
=
\frac{1}{N_j}
\sum_{i=1}^{N}
\gamma_{ij}
(x_i-\mu_j)(x_i-\mu_j)^T
$$


The updated mixing coefficient is:

$$
\pi_j
=
\frac{N_j}{N}
$$

# Task 5 – Python Implementation of Gaussian Mixture Clustering

In [16]:
import numpy as np
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture


np.random.seed(42)


# Generate synthetic data

cluster1 = np.random.normal(
    loc=2,
    scale=0.5,
    size=(100,2)
)

cluster2 = np.random.normal(
    loc=6,
    scale=0.7,
    size=(100,2)
)


X = np.vstack(
    (cluster1,cluster2)
)


# Gaussian Mixture Model

gmm = GaussianMixture(
    n_components=2,
    random_state=42
)


labels = gmm.fit_predict(X)


# Plot clusters

fig = go.Figure()


for i in range(2):

    points = X[labels==i]

    fig.add_trace(
        go.Scatter(
            x=points[:,0],
            y=points[:,1],
            mode="markers",
            name=f"Cluster {i+1}"
        )
    )


fig.update_layout(
    title="Gaussian Mixture Model Clustering",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white"
)


fig.show()

# Task 6 – Conditional Bayesian Updating Interpretation

In Gaussian mixture clustering, the responsibility values act as conditional probabilities.

The posterior probability:

$$
P(z=j|x)
$$


updates the belief about cluster membership after observing the data.


Before observing a data point, the model has only prior knowledge:

$$
P(z=j)=\pi_j
$$


After observing the feature value $x$, the belief is updated:

$$
P(z=j|x)
$$


This follows the Bayesian principle:

$$
\text{Posterior}
\propto
\text{Likelihood}
\times
\text{Prior}
$$


where:


**Prior:**

$$
\pi_j
$$


**Likelihood:**

$$
N(x|\mu_j,\Sigma_j)
$$


**Posterior:**

$$
P(z=j|x)
$$


Therefore, Gaussian mixture clustering can be interpreted as a Bayesian updating process where the observed data modifies the initial probability of cluster membership.


---

# Task 7 – Analysis

The Gaussian Mixture Model successfully separates data into different clusters by estimating the underlying probability distributions.

The EM algorithm improves the clustering iteratively:

1. Initially, cluster parameters are estimated.

2. Responsibilities are calculated.

3. Means, covariance matrices, and mixing probabilities are updated.

4. The process repeats until convergence.


As iterations increase:

- Cluster assignments become more confident.
- The Gaussian distributions better represent the data.
- The likelihood of the observed data increases.


The final responsibility values provide a probabilistic interpretation instead of a simple hard classification.

This makes GMM useful for applications where uncertainty in classification is important, such as anomaly detection, medical diagnosis, and pattern recognition.